<a href="https://colab.research.google.com/github/bsrikanth24/Best-websites-a-programmer-should-visit/blob/master/PySpark_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("Italy", "ITA"),
    ("China", "CHN"),
    ("China", None),
    ("France", "FR"),
    ("Spain", None),
    ("Taiwan", "TWN"),
    ("Taiwan", None)
]

columns = ["Name", "Code"]

df1 = spark.createDataFrame(data, columns)

df1.show()

+------+----+
|  Name|Code|
+------+----+
| Italy| ITA|
| China| CHN|
| China|NULL|
|France|  FR|
| Spain|NULL|
|Taiwan| TWN|
|Taiwan|NULL|
+------+----+



In [13]:
from pyspark.sql import functions as F

df_filtered_one = df1.withColumn("rn", F.expr('row_number() over(partition by Name order by Code desc)')).filter("rn > 1").drop("rn")
df_filtered_one.show(truncate=False)

+------+----+
|Name  |Code|
+------+----+
|China |NULL|
|Taiwan|NULL|
+------+----+



In [12]:
from pyspark.sql import functions as F

df_filtered_greater_than_one = df1.withColumn("rn", F.expr('row_number() over(partition by Name order by Code desc)')).filter('rn = 1').drop("rn")
df_filtered_greater_than_one.show(truncate=False)

+------+----+
|Name  |Code|
+------+----+
|China |CHN |
|France|FR  |
|Italy |ITA |
|Spain |NULL|
|Taiwan|TWN |
+------+----+



In [35]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("A", "A", 1),
    ("A", "A", 2),
    ("A", "A", 3),
    ("A", "B", 4),
    ("A", "B", 5),
    ("A", "C", 6),
    ("A", "D", 7),
    ("A", "E", 8)
]

columns = ["Col1", "Col2", "Col3"]

df2 = spark.createDataFrame(data, columns)

df2.show()

+----+----+----+
|Col1|Col2|Col3|
+----+----+----+
|   A|   A|   1|
|   A|   A|   2|
|   A|   A|   3|
|   A|   B|   4|
|   A|   B|   5|
|   A|   C|   6|
|   A|   D|   7|
|   A|   E|   8|
+----+----+----+



In [36]:
from pyspark.sql import functions as F

df_filtered_one = df2.withColumn("rn", F.expr('row_number() over(partition by Col1, Col2 order by Col3 desc)')).filter("rn > 1").drop("rn")
df_filtered_one.show(truncate=False)

+----+----+----+
|Col1|Col2|Col3|
+----+----+----+
|A   |A   |2   |
|A   |A   |1   |
|A   |B   |4   |
+----+----+----+



In [37]:
from pyspark.sql import functions as F

df_filtered_one = df2.withColumn("rn", F.expr('row_number() over(partition by Col1, Col2 order by Col3 desc)')).filter("rn = 1").drop("rn")
df_filtered_one.show(truncate=False)

+----+----+----+
|Col1|Col2|Col3|
+----+----+----+
|A   |A   |3   |
|A   |B   |5   |
|A   |C   |6   |
|A   |D   |7   |
|A   |E   |8   |
+----+----+----+



In [38]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("Col1", "Col2")

# 2. Add a count column, filter for duplicates, and drop the count column
df_all_duplicates = (df2
                     .withColumn("cnt", F.count("*").over(window_spec))
                     .filter(F.col("cnt") > 1)
                     .drop("cnt")
)
df_all_duplicates.show()

+----+----+----+
|Col1|Col2|Col3|
+----+----+----+
|   A|   A|   1|
|   A|   A|   2|
|   A|   A|   3|
|   A|   B|   4|
|   A|   B|   5|
+----+----+----+



In [47]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("Col1", "Col2")

# 2. Add a count column, filter for duplicates, and drop the count column
df_all_duplicates = (df2
                     .withColumn("cnt", F.count("*").over(window_spec))
                     .filter(F.col("cnt") == 1)
                    .drop("cnt")
)

df_all_duplicates.show()

+----+----+----+
|Col1|Col2|Col3|
+----+----+----+
|   A|   C|   6|
|   A|   D|   7|
|   A|   E|   8|
+----+----+----+



In [33]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("John", "IT", 50000),
    ("Alice", "HR", 60000),
    ("Bob", "IT", 75000),
    ("David", "Finance", 80000),
    ("Sarah", "HR", 65000)
]

columns = ["employee", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

+--------+----------+------+
|employee|department|salary|
+--------+----------+------+
|    John|        IT| 50000|
|   Alice|        HR| 60000|
|     Bob|        IT| 75000|
|   David|   Finance| 80000|
|   Sarah|        HR| 65000|
+--------+----------+------+



In [34]:
df = (
    df.withColumn('rank', F.expr('rank() over(partition By department order By salary desc)'))
    .filter('rank = 1')
    .drop('rank'))

df.show()

+--------+----------+------+
|employee|department|salary|
+--------+----------+------+
|   David|   Finance| 80000|
|   Sarah|        HR| 65000|
|     Bob|        IT| 75000|
+--------+----------+------+

